# Notebook_2_SQL_Gold_Load — Gold KPIs (Daily Grain)
### O&G Rig Operations Intelligence Platform
**Layer:** Silver → Gold  
**Run after:** Notebook_1_SQL_Silver_Load  
**Strategy:** Watermark-based incremental MERGE on date_id  
**Grain:** Daily — Power BI aggregates up to week/month/quarter/year via dim_date  

| # | Table | Grain |
|---|---|---|
| 1 | gold.rig_performance_summary | rig_id × date_id |
| 2 | gold.equipment_reliability | equipment_type × region_name × date_id |
| 3 | gold.sla_compliance | equipment_type × region_name × date_id |
| 4 | gold.maintenance_summary | rig_id × date_id |
| 5 | gold.regional_benchmarks | region_name × date_id |

In [0]:
-- CELL 1 — SETUP
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
-- CELL 2 — CREATE GOLD TABLES (one-time setup)
-- Daily grain — date_id is the FK to dim_date
-- CREATE TABLE IF NOT EXISTS — safe to re-run, never drops data

CREATE TABLE IF NOT EXISTS workspace.gold.rig_performance_summary (
    date_id                     INT,
    rig_id                      STRING,
    rig_name                    STRING,
    region_name                 STRING,
    rig_type                    STRING,
    well_name                   STRING,
    well_type                   STRING,
    rig_status                  STRING,
    drilling_hours              DOUBLE,
    downtime_hours              DOUBLE,
    npt_hours                   DOUBLE,
    rop_ft_hr                   DOUBLE,
    cost_per_day                DOUBLE,
    uptime_pct                  DOUBLE,
    downtime_pct                DOUBLE,
    npt_pct                     DOUBLE,
    sla_met                     INT,
    is_sla_breach               INT,
    is_controllable_downtime    BOOLEAN,
    total_incidents             BIGINT,
    serious_incidents           BIGINT,
    recordable_incidents        BIGINT,
    incident_downtime_hrs       DOUBLE,
    total_work_orders           BIGINT,
    planned_wo                  BIGINT,
    unplanned_wo                BIGINT,
    total_maint_cost            DOUBLE,
    total_maint_hrs             DOUBLE,
    distinct_crew_count         BIGINT,
    total_hours_worked          DOUBLE,
    total_overtime_hours        DOUBLE,
    loaded_at                   TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.equipment_reliability (
    date_id                         INT,
    equipment_type                  STRING,
    region_name                     STRING,
    active_rig_count                BIGINT,
    total_events                    BIGINT,
    failure_count                   BIGINT,
    downtime_event_count            BIGINT,
    total_downtime_hours            DOUBLE,
    avg_downtime_per_event          DOUBLE,
    avg_resolution_hrs              DOUBLE,
    sla_met_count                   BIGINT,
    sla_breach_count                BIGINT,
    sla_compliance_pct              DOUBLE,
    loaded_at                       TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.sla_compliance (
    date_id                         INT,
    equipment_type                  STRING,
    region_name                     STRING,
    total_events                    BIGINT,
    sla_met_count                   BIGINT,
    sla_breach_count                BIGINT,
    sla_compliance_pct              DOUBLE,
    failure_count                   BIGINT,
    avg_resolution_hrs              DOUBLE,
    total_work_orders               BIGINT,
    planned_wo                      BIGINT,
    unplanned_wo                    BIGINT,
    total_cost                      DOUBLE,
    avg_duration_hrs                DOUBLE,
    loaded_at                       TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.maintenance_summary (
    date_id                     INT,
    rig_id                      STRING,
    rig_name                    STRING,
    region_name                 STRING,
    distinct_equipment_types    BIGINT,
    total_work_orders           BIGINT,
    planned_wo                  BIGINT,
    unplanned_wo                BIGINT,
    preventive_wo               BIGINT,
    high_cost_wo                BIGINT,
    long_duration_wo            BIGINT,
    total_cost                  DOUBLE,
    avg_cost_per_wo             DOUBLE,
    total_duration_hrs          DOUBLE,
    avg_duration_hrs            DOUBLE,
    loaded_at                   TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.regional_benchmarks (
    date_id                     INT,
    region_name                 STRING,
    active_rig_count            BIGINT,
    avg_uptime_pct              DOUBLE,
    avg_downtime_pct            DOUBLE,
    avg_npt_pct                 DOUBLE,
    avg_rop_ft_hr               DOUBLE,
    avg_cost_per_day            DOUBLE,
    total_incidents             BIGINT,
    serious_incidents           BIGINT,
    recordable_incidents        BIGINT,
    incident_rate_per_rig       DOUBLE,
    total_work_orders           BIGINT,
    unplanned_work_orders       BIGINT,
    unplanned_maint_pct         DOUBLE,
    total_maint_cost            DOUBLE,
    total_sla_breaches          BIGINT,
    total_sla_met               BIGINT,
    sla_compliance_pct          DOUBLE,
    total_downtime_hours        DOUBLE,
    total_failures              BIGINT,
    score_uptime                DOUBLE,
    score_safety                DOUBLE,
    score_sla                   DOUBLE,
    composite_score             DOUBLE,
    performance_rank            BIGINT,
    loaded_at                   TIMESTAMP
) USING DELTA;

SELECT 'Gold tables ready' AS status;

In [0]:
-- CELL 3 — gold.rig_performance_summary
-- Grain   : rig_id × date_id (daily)
-- Sources : silver_rig_ops + silver_incident_log
--           + silver_maintenance_log + silver_crew_hours

MERGE INTO workspace.gold.rig_performance_summary AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(date_id), 0) AS wm
        FROM workspace.gold.rig_performance_summary
    ),
    rig_agg AS (
        SELECT
            date_id,
            rig_id,
            rig_name,
            region_name,
            rig_type,
            well_name,
            well_type,
            rig_status,
            drilling_hours_clean                AS drilling_hours,
            downtime_hours,
            npt_hours,
            rop_ft_hr_clean                     AS rop_ft_hr,
            cost_per_day_clean                  AS cost_per_day,
            uptime_pct,
            downtime_pct,
            npt_pct,
            sla_met,
            is_sla_breach,
            is_controllable_downtime
        FROM workspace.silver.silver_rig_ops
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
    ),
    inc_agg AS (
        SELECT
            date_id,
            rig_id,
            COUNT(*)                            AS total_incidents,
            SUM(is_serious_incident)            AS serious_incidents,
            SUM(is_recordable)                  AS recordable_incidents,
            ROUND(SUM(downtime_caused_hrs), 2)  AS incident_downtime_hrs
        FROM workspace.silver.silver_incident_log
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, rig_id
    ),
    maint_agg AS (
        SELECT
            date_id,
            rig_id,
            COUNT(*)                                        AS total_work_orders,
            SUM(CAST(is_planned AS INT))                    AS planned_wo,
            SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_wo,
            ROUND(SUM(cost), 2)                             AS total_maint_cost,
            ROUND(SUM(duration_hrs), 2)                     AS total_maint_hrs
        FROM workspace.silver.silver_maintenance_log
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, rig_id
    ),
    crew_agg AS (
        SELECT
            date_id,
            rig_id,
            COUNT(DISTINCT crew_id)                         AS distinct_crew_count,
            ROUND(SUM(hours_worked), 2)                     AS total_hours_worked,
            ROUND(SUM(overtime_hours), 2)                   AS total_overtime_hours
        FROM workspace.silver.silver_crew_hours
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, rig_id
    )
    SELECT
        r.date_id,
        r.rig_id,
        r.rig_name,
        r.region_name,
        r.rig_type,
        r.well_name,
        r.well_type,
        r.rig_status,
        r.drilling_hours,
        r.downtime_hours,
        r.npt_hours,
        r.rop_ft_hr,
        r.cost_per_day,
        r.uptime_pct,
        r.downtime_pct,
        r.npt_pct,
        r.sla_met,
        r.is_sla_breach,
        r.is_controllable_downtime,
        COALESCE(i.total_incidents, 0)          AS total_incidents,
        COALESCE(i.serious_incidents, 0)        AS serious_incidents,
        COALESCE(i.recordable_incidents, 0)     AS recordable_incidents,
        COALESCE(i.incident_downtime_hrs, 0.0)  AS incident_downtime_hrs,
        COALESCE(m.total_work_orders, 0)        AS total_work_orders,
        COALESCE(m.planned_wo, 0)               AS planned_wo,
        COALESCE(m.unplanned_wo, 0)             AS unplanned_wo,
        COALESCE(m.total_maint_cost, 0.0)       AS total_maint_cost,
        COALESCE(m.total_maint_hrs, 0.0)        AS total_maint_hrs,
        COALESCE(c.distinct_crew_count, 0)      AS distinct_crew_count,
        COALESCE(c.total_hours_worked, 0.0)     AS total_hours_worked,
        COALESCE(c.total_overtime_hours, 0.0)   AS total_overtime_hours,
        current_timestamp()                     AS loaded_at
    FROM rig_agg r
    LEFT JOIN inc_agg   i ON r.rig_id = i.rig_id AND r.date_id = i.date_id
    LEFT JOIN maint_agg m ON r.rig_id = m.rig_id AND r.date_id = m.date_id
    LEFT JOIN crew_agg  c ON r.rig_id = c.rig_id AND r.date_id = c.date_id
) AS source
ON  target.rig_id   = source.rig_id
AND target.date_id  = source.date_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT 'rig_performance_summary' AS table_name, COUNT(*) AS row_count,
       MIN(date_id) AS min_date, MAX(date_id) AS max_date
FROM workspace.gold.rig_performance_summary;

In [0]:
-- CELL 4 — gold.equipment_reliability
-- Grain  : equipment_type × region_name × date_id (daily)
-- Source : silver_equipment_events

MERGE INTO workspace.gold.equipment_reliability AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(date_id), 0) AS wm
        FROM workspace.gold.equipment_reliability
    )
    SELECT
        date_id,
        equipment_type,
        region_name,
        COUNT(DISTINCT rig_id)                          AS active_rig_count,
        COUNT(*)                                        AS total_events,
        SUM(failure_flag)                               AS failure_count,
        SUM(CASE WHEN downtime_caused_hrs > 0
            THEN 1 ELSE 0 END)                          AS downtime_event_count,
        ROUND(SUM(downtime_caused_hrs), 2)              AS total_downtime_hours,
        ROUND(AVG(CASE WHEN downtime_caused_hrs > 0
            THEN downtime_caused_hrs END), 2)           AS avg_downtime_per_event,
        ROUND(AVG(resolution_hrs), 2)                   AS avg_resolution_hrs,
        SUM(sla_met)                                    AS sla_met_count,
        SUM(is_sla_breach)                              AS sla_breach_count,
        ROUND(
            CASE WHEN (SUM(sla_met) + SUM(is_sla_breach)) > 0
            THEN SUM(sla_met) / (SUM(sla_met) + SUM(is_sla_breach)) * 100
            ELSE NULL END, 2)                           AS sla_compliance_pct,
        current_timestamp()                             AS loaded_at
    FROM workspace.silver.silver_equipment_events
    WHERE is_valid = 1
    AND date_id > (SELECT wm FROM watermark)
    GROUP BY date_id, equipment_type, region_name
) AS source
ON  target.equipment_type = source.equipment_type
AND target.region_name    = source.region_name
AND target.date_id        = source.date_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT 'equipment_reliability' AS table_name, COUNT(*) AS row_count,
       MIN(date_id) AS min_date, MAX(date_id) AS max_date
FROM workspace.gold.equipment_reliability;

In [0]:
-- CELL 5 — gold.sla_compliance
-- Grain  : equipment_type × region_name × date_id (daily)
-- Sources: silver_equipment_events + silver_maintenance_log

MERGE INTO workspace.gold.sla_compliance AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(date_id), 0) AS wm
        FROM workspace.gold.sla_compliance
    ),
    sla_agg AS (
        SELECT
            date_id,
            equipment_type,
            region_name,
            COUNT(*)                                AS total_events,
            SUM(sla_met)                            AS sla_met_count,
            SUM(is_sla_breach)                      AS sla_breach_count,
            SUM(failure_flag)                       AS failure_count,
            ROUND(AVG(resolution_hrs), 2)           AS avg_resolution_hrs
        FROM workspace.silver.silver_equipment_events
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, equipment_type, region_name
    ),
    maint_agg AS (
        SELECT
            date_id,
            equipment_type,
            region_name,
            COUNT(*)                                        AS total_work_orders,
            SUM(CAST(is_planned AS INT))                    AS planned_wo,
            SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_wo,
            ROUND(SUM(cost), 2)                             AS total_cost,
            ROUND(AVG(duration_hrs), 2)                     AS avg_duration_hrs
        FROM workspace.silver.silver_maintenance_log
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, equipment_type, region_name
    )
    SELECT
        s.date_id,
        s.equipment_type,
        s.region_name,
        s.total_events,
        COALESCE(s.sla_met_count, 0)                AS sla_met_count,
        COALESCE(s.sla_breach_count, 0)             AS sla_breach_count,
        ROUND(
            CASE WHEN (s.sla_met_count + s.sla_breach_count) > 0
            THEN s.sla_met_count / (s.sla_met_count + s.sla_breach_count) * 100
            ELSE NULL END, 2)                       AS sla_compliance_pct,
        s.failure_count,
        s.avg_resolution_hrs,
        COALESCE(m.total_work_orders, 0)            AS total_work_orders,
        COALESCE(m.planned_wo, 0)                   AS planned_wo,
        COALESCE(m.unplanned_wo, 0)                 AS unplanned_wo,
        COALESCE(m.total_cost, 0.0)                 AS total_cost,
        COALESCE(m.avg_duration_hrs, 0.0)           AS avg_duration_hrs,
        current_timestamp()                         AS loaded_at
    FROM sla_agg s
    LEFT JOIN maint_agg m
        ON  s.equipment_type = m.equipment_type
        AND s.region_name    = m.region_name
        AND s.date_id        = m.date_id
) AS source
ON  target.equipment_type = source.equipment_type
AND target.region_name    = source.region_name
AND target.date_id        = source.date_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT 'sla_compliance' AS table_name, COUNT(*) AS row_count,
       MIN(date_id) AS min_date, MAX(date_id) AS max_date
FROM workspace.gold.sla_compliance;

In [0]:
-- CELL 6 — gold.maintenance_summary
-- Grain  : rig_id × date_id (daily)
-- Source : silver_maintenance_log
-- Note   : only days with work orders are written — no empty rows

MERGE INTO workspace.gold.maintenance_summary AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(date_id), 0) AS wm
        FROM workspace.gold.maintenance_summary
    )
    SELECT
        date_id,
        rig_id,
        rig_name,
        region_name,
        COUNT(DISTINCT equipment_type)                  AS distinct_equipment_types,
        COUNT(*)                                        AS total_work_orders,
        SUM(CAST(is_planned AS INT))                    AS planned_wo,
        SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_wo,
        SUM(is_preventive)                              AS preventive_wo,
        SUM(is_high_cost)                               AS high_cost_wo,
        SUM(is_long_maintenance)                        AS long_duration_wo,
        ROUND(SUM(cost), 2)                             AS total_cost,
        ROUND(AVG(cost), 2)                             AS avg_cost_per_wo,
        ROUND(SUM(duration_hrs), 2)                     AS total_duration_hrs,
        ROUND(AVG(duration_hrs), 2)                     AS avg_duration_hrs,
        current_timestamp()                             AS loaded_at
    FROM workspace.silver.silver_maintenance_log
    WHERE is_valid = 1
    AND date_id > (SELECT wm FROM watermark)
    GROUP BY date_id, rig_id, rig_name, region_name
) AS source
ON  target.rig_id   = source.rig_id
AND target.date_id  = source.date_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT 'maintenance_summary' AS table_name, COUNT(*) AS row_count,
       MIN(date_id) AS min_date, MAX(date_id) AS max_date
FROM workspace.gold.maintenance_summary;

In [0]:
-- CELL 7 — gold.regional_benchmarks
-- Grain  : region_name × date_id (daily)
-- Source : silver_rig_ops + silver_incident_log
--          + silver_maintenance_log + silver_equipment_events
-- Composite score: 40% uptime + 20% safety + 20% SLA + 20% availability

MERGE INTO workspace.gold.regional_benchmarks AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(date_id), 0) AS wm
        FROM workspace.gold.regional_benchmarks
    ),
    rig_agg AS (
        SELECT
            date_id,
            region_name,
            COUNT(DISTINCT rig_id)                      AS active_rig_count,
            ROUND(AVG(uptime_pct), 2)                   AS avg_uptime_pct,
            ROUND(AVG(downtime_pct), 2)                 AS avg_downtime_pct,
            ROUND(AVG(npt_pct), 2)                      AS avg_npt_pct,
            ROUND(AVG(rop_ft_hr), 2)                    AS avg_rop_ft_hr,
            ROUND(AVG(cost_per_day), 2)                 AS avg_cost_per_day,
            SUM(sla_met)                                AS total_sla_met,
            SUM(is_sla_breach)                          AS total_sla_breaches
        FROM workspace.silver.silver_rig_ops
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, region_name
    ),
    inc_agg AS (
        SELECT
            date_id,
            region_name,
            COUNT(*)                                    AS total_incidents,
            SUM(is_serious_incident)                    AS serious_incidents,
            SUM(is_recordable)                          AS recordable_incidents
        FROM workspace.silver.silver_incident_log
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, region_name
    ),
    maint_agg AS (
        SELECT
            date_id,
            region_name,
            COUNT(*)                                        AS total_work_orders,
            SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_work_orders,
            ROUND(SUM(cost), 2)                             AS total_maint_cost
        FROM workspace.silver.silver_maintenance_log
        WHERE is_valid = 1
        AND date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, region_name
    ),
    equip_agg AS (
        SELECT
            date_id,
            region_name,
            ROUND(SUM(total_downtime_hours), 2)         AS total_downtime_hours,
            SUM(failure_count)                          AS total_failures,
            ROUND(AVG(sla_compliance_pct), 2)           AS avg_sla_compliance_pct
        FROM workspace.gold.equipment_reliability
        WHERE date_id > (SELECT wm FROM watermark)
        GROUP BY date_id, region_name
    ),
    scored AS (
        SELECT
            r.date_id,
            r.region_name,
            r.active_rig_count,
            r.avg_uptime_pct,
            r.avg_downtime_pct,
            r.avg_npt_pct,
            r.avg_rop_ft_hr,
            r.avg_cost_per_day,
            COALESCE(i.total_incidents, 0)              AS total_incidents,
            COALESCE(i.serious_incidents, 0)            AS serious_incidents,
            COALESCE(i.recordable_incidents, 0)         AS recordable_incidents,
            ROUND(
                COALESCE(i.total_incidents, 0)
                / NULLIF(r.active_rig_count, 0), 3)     AS incident_rate_per_rig,
            COALESCE(m.total_work_orders, 0)            AS total_work_orders,
            COALESCE(m.unplanned_work_orders, 0)        AS unplanned_work_orders,
            ROUND(
                CASE WHEN COALESCE(m.total_work_orders,0) > 0
                THEN COALESCE(m.unplanned_work_orders, 0)
                    / m.total_work_orders * 100
                ELSE 0 END, 2)                          AS unplanned_maint_pct,
            COALESCE(m.total_maint_cost, 0.0)           AS total_maint_cost,
            r.total_sla_breaches,
            r.total_sla_met,
            ROUND(
                CASE WHEN (r.total_sla_met + r.total_sla_breaches) > 0
                THEN r.total_sla_met
                    / (r.total_sla_met + r.total_sla_breaches) * 100
                ELSE NULL END, 2)                       AS sla_compliance_pct,
            COALESCE(e.total_downtime_hours, 0.0)       AS total_downtime_hours,
            COALESCE(e.total_failures, 0)               AS total_failures,
            -- Score components
            COALESCE(r.avg_uptime_pct, 0.0)             AS score_uptime,
            ROUND(100 - COALESCE(
                CASE WHEN COALESCE(i.total_incidents,0) > 0
                THEN COALESCE(i.serious_incidents,0)
                    / i.total_incidents * 100
                ELSE 0 END, 0.0), 2)                    AS score_safety,
            COALESCE(e.avg_sla_compliance_pct, 0.0)     AS score_sla,
            -- Composite score
            ROUND(
                COALESCE(r.avg_uptime_pct, 0.0)         * 0.40 +
                ROUND(100 - COALESCE(
                    CASE WHEN COALESCE(i.total_incidents,0) > 0
                    THEN COALESCE(i.serious_incidents,0)
                        / i.total_incidents * 100
                    ELSE 0 END, 0.0), 2)                * 0.20 +
                COALESCE(e.avg_sla_compliance_pct, 0.0) * 0.20 +
                COALESCE(r.avg_uptime_pct, 0.0)         * 0.20,
            2)                                          AS composite_score
        FROM rig_agg r
        LEFT JOIN inc_agg   i ON r.region_name = i.region_name AND r.date_id = i.date_id
        LEFT JOIN maint_agg m ON r.region_name = m.region_name AND r.date_id = m.date_id
        LEFT JOIN equip_agg e ON r.region_name = e.region_name AND r.date_id = e.date_id
    )
    SELECT
        *,
        RANK() OVER (
            PARTITION BY date_id
            ORDER BY composite_score DESC
        )                                               AS performance_rank,
        current_timestamp()                             AS loaded_at
    FROM scored
) AS source
ON  target.region_name = source.region_name
AND target.date_id     = source.date_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT 'regional_benchmarks' AS table_name, COUNT(*) AS row_count,
       MIN(date_id) AS min_date, MAX(date_id) AS max_date
FROM workspace.gold.regional_benchmarks;

In [0]:
-- CELL 8 — FINAL VALIDATION

SELECT 'rig_performance_summary' AS table_name, COUNT(*) AS row_count,
       MIN(date_id) AS min_date, MAX(date_id) AS max_date
FROM workspace.gold.rig_performance_summary
UNION ALL
SELECT 'equipment_reliability', COUNT(*), MIN(date_id), MAX(date_id)
FROM workspace.gold.equipment_reliability
UNION ALL
SELECT 'sla_compliance', COUNT(*), MIN(date_id), MAX(date_id)
FROM workspace.gold.sla_compliance
UNION ALL
SELECT 'maintenance_summary', COUNT(*), MIN(date_id), MAX(date_id)
FROM workspace.gold.maintenance_summary
UNION ALL
SELECT 'regional_benchmarks', COUNT(*), MIN(date_id), MAX(date_id)
FROM workspace.gold.regional_benchmarks
ORDER BY table_name;

In [0]:
-- Regional benchmarks latest day — performance rankings
SELECT
    performance_rank,
    region_name,
    composite_score,
    score_uptime,
    score_safety,
    score_sla,
    avg_uptime_pct,
    incident_rate_per_rig,
    sla_compliance_pct
FROM workspace.gold.regional_benchmarks
WHERE date_id = (SELECT MAX(date_id) FROM workspace.gold.regional_benchmarks)
ORDER BY performance_rank;